<a href="https://colab.research.google.com/github/KuchkorovBoburbek/computer_vision/blob/main/menu_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
print("Menu dedector")

Menu dedector


#  Menu detector loyihasi bu taomlar rasmlari orqali train qilib malum taomlarni taniydigan model

In [12]:
#---------------------
# Import Libraries
#---------------------
import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import drive
import torchvision.transforms as transforms

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

from torchvision.models import mobilenet_v2



In [4]:
drive.mount('/content/drive') # drive ga ulanamiz

Mounted at /content/drive


In [5]:
# Define dataset path
DATASET_PATH = '/content/drive/MyDrive/Food_101_dataset'


CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert', # label grouping  |  class consolidation
    'kebeb': 'kebeb',
    'pilav': 'pilaf',
    'ice_cream': 'dessert'     # label grouping  |  class consolidation
}





CLASSES = ['hamburger', 'dessert', 'hot_dog', 'kebeb', 'pilaf' ]
CLASS_TO_INDEX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(f'Classes: {CLASSES}')
print(f'Class to index: {CLASS_TO_INDEX}')
print(f'Number of classes: {NUM_CLASSES}')

# bu bizning ishlarimiz ketmaketligini belgilaydi
transform = transforms.Compose([
    transforms.Resize((224, 224)) , # bu rasmlarimiz sizeni o'zgartiradi
    transforms.ToTensor(), # tensorga o'giramiz(raqamlarga)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# Tensorga o'girgach u flot korinishiga otadi 0.0 ~ 1.0
# H, W, C => C, H, W  | => channel : RGB
# Normalize => hamma channellarni qayta sozlab beradi
# Normalize : pixel = (pixel-mean)/std



print("transform:", transform)


Classes: ['hamburger', 'dessert', 'hot_dog', 'kebeb', 'pilaf']
Class to index: {'hamburger': 0, 'dessert': 1, 'hot_dog': 2, 'kebeb': 3, 'pilaf': 4}
Number of classes: 5
transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [6]:
# -------------------------
# Custom Dataset Class
# -------------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
# umumiy filelarni o'qish vaqtida data uzunligini olib beradi
    def __len__(self):
        print('images_length', len(self.images))
        return len(self.images)
#
    def __getitem__(self, idx):
        img_path = self.images[idx]
        print('image_path', img_path)

        label = self.labels[idx]
        print('label', label)
# agar rasmda RGB bolmasa convert qilamiz
        try:
            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}") # rasm filelarda kamchilik bolsa skip qilish
            return self.__getitem__((idx + 1) % len(self.images))

        if self.transform:
            image = self.transform(image)

        return image, label

In [7]:
# -------------------------
# Gather and Split Data
# -------------------------

all_images = [] # created empty list

for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)  # /content/drive/MyDrive/food_101_dataset/hamburber
    print('class_path:', class_path)

    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue

    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img) # /content/drive/MyDrive/food_101_dataset/hamburber/233.jpg
            all_images.append((full_path, CLASS_TO_INDEX[mapped_class])) # Changed CLASS_TO_IDX to CLASS_TO_INDEX

np.random.shuffle(all_images)

split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))

img, lbl = dataset[0]

class_path: /content/drive/MyDrive/Food_101_dataset/hamburger
class_path: /content/drive/MyDrive/Food_101_dataset/hot_dog
class_path: /content/drive/MyDrive/Food_101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/Food_101_dataset/kebeb
class_path: /content/drive/MyDrive/Food_101_dataset/pilav
class_path: /content/drive/MyDrive/Food_101_dataset/ice_cream
all_images: [('/content/drive/MyDrive/Food_101_dataset/ice_cream/3229615.jpg', 1), ('/content/drive/MyDrive/Food_101_dataset/ice_cream/2598336.jpg', 1), ('/content/drive/MyDrive/Food_101_dataset/hamburger/1343040.jpg', 0), ('/content/drive/MyDrive/Food_101_dataset/hot_dog/3265229.jpg', 2), ('/content/drive/MyDrive/Food_101_dataset/ice_cream/3773707.jpg', 1), ('/content/drive/MyDrive/Food_101_dataset/chocolate_cake/1805931.jpg', 1), ('/content/drive/MyDrive/Food_101_dataset/hot_dog/1210977.jpg', 2), ('/content/drive/MyDrive/Food_101_dataset/ice_cream/2040874.jpg', 1), ('/content/drive/MyDrive/Food_101_dataset/chocolate_cake/86

In [8]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2) # thread 2ta
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)


images_length 3224
images_length 3224


In [10]:
# pretrained model : bu mavjud modelni olib ishlatish
model = mobilenet_v2(weights='IMAGENET1K_V1') #pretrained model | CNN | 1000dan ortiq class | mln ortiq rasm orqali train
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
# fine-tuning, | backbone |model layer freeze

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 131MB/s]


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("devive", device)
model.to(device)


devive cuda


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [14]:
criterion = nn.CrossEntropyLoss() # Loss functionni tanlaymiz | 70% burger
optimizer = optim.Adam(model.parameters(), lr=0.001) # parametr update
test = torch.backends.cudnn.benchmark = True # banchmark setting = trick
# GPU bilan ishlashda eng muqobil algoritmni tanlab ishlaydi